# UrbanPulse — Notebook 03: Model Training

Bible §5 Notebook 03. Logic lives in `train.py` + `modeling.py`.

**Framing (decision #12):** +15 min forecast (`horizon=3`). The label is congestion 15 min ahead, so current occupancy/queue are legitimate predictors — no leakage. Temporal split: days 1-10 train / 11-12 val / 13-14 test, with a cross-split guard in `modeling.shift_target`. LINK_ID is target-encoded on the train split only.

**Produces:** `models/*.pkl`, `reports/model_metrics.csv`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import train, modeling, io_utils, config
import pandas as pd

## 1. Leak-safe data prep (shift + split + encode)

In [2]:
df = io_utils.load_parquet(config.FEATURES_PARQUET)
x_tr, x_va, x_te, y = modeling.prepare_xy(df, horizon=config.HORIZON_INTERVALS, leak_free=False)
print('train', x_tr.shape, 'val', x_va.shape, 'test', x_te.shape)
print('train pos-rate', round(y['train'].mean(),3))

train (189882, 24) val (37818, 24) test (37818, 24)
train pos-rate 0.132


## 2. Train all 7 models (skips if already trained)

In [3]:
if config.MODEL_METRICS_CSV.exists():
    metrics = pd.read_csv(config.MODEL_METRICS_CSV)
    print('loaded cached metrics')
else:
    metrics = train.train_all()
metrics

loaded cached metrics


,model,accuracy,precision,recall,f1_weighted,f1_macro,roc_auc,pr_auc,train_time_s,infer_ms_per_row
0,catboost,0.9404,0.7510,0.8013,0.9412,0.8705,0.9770,0.8574,15.95,0.0005
1,xgboost,0.9397,0.7489,0.7974,0.9405,0.8688,0.9764,0.8543,5.83,0.0051
2,lightgbm,0.9391,0.7459,0.7967,0.9400,0.8677,0.9762,0.8480,5.08,0.0076
3,random_forest,0.9388,0.7494,0.7860,0.9395,0.8660,0.9757,0.8469,102.33,0.0190
4,gradient_boosting,0.9390,0.7462,0.7945,0.9398,0.8672,0.9752,0.8472,74.11,0.0007
5,extra_trees,0.9376,0.7482,0.7747,0.9381,0.8627,0.9743,0.8438,32.25,0.0242
6,decision_tree,0.9330,0.7191,0.7842,0.9342,0.8558,0.9482,0.7552,2.65,0.0001


## 3. Gate — all ROC-AUC > 0.85, inference < 500 ms/row

In [4]:
print('models:', len(metrics))
print('all ROC-AUC > 0.85:', bool((metrics.roc_auc>0.85).all()))
print('all infer < 500ms/row:', bool((metrics.infer_ms_per_row<500).all()))

models: 7
all ROC-AUC > 0.85: True
all infer < 500ms/row: True


## 4. Ranking by ROC-AUC and PR-AUC
PR-AUC is the metric that matters at 13% positive — accuracy is misleading.

In [5]:
metrics.sort_values('pr_auc', ascending=False)[['model','roc_auc','pr_auc','precision','recall','train_time_s']]

,model,roc_auc,pr_auc,precision,recall,train_time_s
0,catboost,0.9770,0.8574,0.7510,0.8013,15.95
1,xgboost,0.9764,0.8543,0.7489,0.7974,5.83
2,lightgbm,0.9762,0.8480,0.7459,0.7967,5.08
4,gradient_boosting,0.9752,0.8472,0.7462,0.7945,74.11
3,random_forest,0.9757,0.8469,0.7494,0.7860,102.33
5,extra_trees,0.9743,0.8438,0.7482,0.7747,32.25
6,decision_tree,0.9482,0.7552,0.7191,0.7842,2.65
